# Clase 2 — Convolución y Filtros

## El reto

Te piden instrumentar la cámara de una academia de fútbol para un
sistema IoT: una placa tipo Raspberry Pi con una cámara barata, mirando
la cancha, que debe **encontrar automáticamente el contorno del balón y
del jugador** en cada cuadro — el primer paso para medir posesión,
disparos, lo que sea que venga después.

El problema: la cámara es barata. Poca luz, sensor con ruido, imágenes
que nunca son tan limpias como las del laboratorio. Si le pides a
`cv2.findContours` que trabaje directo sobre eso, encuentra ruido, no
un balón.

Este notebook resuelve ese reto de punta a punta, en cuatro actos —los
mismos de `VisionClase2_Convolucion.html`— empezando por lo más simple
y terminando con **una sola operación**, aplicada una y otra vez con
distintos pesos: la convolución.

- **Acto 1** — antes de vecindarios: aritmética simple entre dos
  imágenes completas (sumar, restar, mezclar, aclarar/oscurecer).
- **Acto 2** — descubrimos qué es la convolución y la construimos desde
  cero, a mano, para saber exactamente qué hace por dentro antes de
  confiar en la versión rápida de OpenCV.
- **Acto 3** — la cámara mete ruido; aprendemos a limpiarlo sin borrar
  el balón.
- **Acto 4** — con la imagen limpia, medimos dónde cambia de golpe la
  intensidad — ahí están los bordes — y cerramos el reto con contornos
  detectados sobre una foto real.

Dos imágenes acompañan todo el recorrido:

- `assets/img/image_messi.png` — nuestra "cámara de cancha": una foto
  real, con textura y ruido de sensor.
- `assets/img/escena.png` — una escena sintética y limpia, que usamos
  como control: si un filtro falla incluso ahí, el problema es el
  filtro, no la imagen.

> Nota de rendimiento: casi todo aquí usa OpenCV (vectorizado en C++,
> muy rápido). La única excepción a propósito es el Acto 2, donde
> construimos la convolución a mano, con loops de Python — lento, pero
> es la única forma de ver el mecanismo por dentro antes de confiar en
> la versión rápida.

**Este notebook es también tu primera guía de OpenCV.** Es normal que
sea la primera vez que usas la librería: cada vez que aparece una
función nueva (`cv2.algo`), hay un recuadro **🔧 OpenCV** justo antes
que explica qué hace, qué parámetros recibe y qué devuelve — no hace
falta memorizarlas, solo entender el patrón para poder buscarlas luego
en la documentación. Al final hay una chuleta con todas juntas.

**Sobre el código:** cada celda hace **una sola cosa** (importar,
definir, calcular, o mostrar) — corre el notebook celda por celda, sin
saltarte ninguna.


## 0 · Antes de empezar

El equipo para la misión: las librerías, unos helpers para mostrar
imágenes sin repetir código, y las dos "cámaras" (imágenes) con las que
vamos a trabajar durante todo el recorrido.

### 0.1 · Qué es OpenCV

**OpenCV** (*Open Computer Vision*) es una librería de visión artificial
escrita en C++, con bindings a Python — por eso se instala como
`opencv-python` (`pip install opencv-python`) pero se importa como
`cv2` (el nombre viene de la versión 2 de la API en C, que es la que
Python terminó adoptando).

Tres ideas que vas a usar todo el notebook, y todo el semestre:

1. **Una imagen es un array de NumPy.** OpenCV no tiene un "tipo Imagen"
   propio — lee un archivo y te devuelve un `numpy.ndarray`. Todo lo que
   ya sabes de NumPy (slicing, `.shape`, operaciones vectorizadas) aplica
   directo. Un array de forma `(alto, ancho)` es una imagen en escala de
   grises; `(alto, ancho, 3)` es a color.
2. **Los píxeles casi siempre son `uint8`.** Enteros de 0 a 255 por
   canal. Ojo con esto: si haces `img + 10` y el resultado pasa de 255,
   **da la vuelta** (256 se vuelve 0) en vez de saturar — el Acto 1 lo
   muestra con números concretos, y es la razón por la que casi todo el
   notebook usa `.astype(np.float64)` antes de sumar o restar cuando el
   resultado puede salirse de 0..255, volviendo a `uint8` al final (con
   `norm8`, más abajo).
3. **OpenCV usa BGR, no RGB.** Por razones históricas, una imagen a color
   en OpenCV guarda los canales en el orden Azul-Verde-Rojo, al revés de
   como los espera casi todo lo demás (matplotlib, PIL, los navegadores).
   Es el error más común de quien empieza con OpenCV: mostrar una imagen
   "cruda" con matplotlib y que salga con los colores invertidos. Por
   eso el helper `show()` de la celda de abajo hace
   `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` antes de dibujar — conviene
   entenderlo ahora para no tropezar con esto más adelante.

In [ ]:
# Librerías: cv2 (OpenCV) para procesar, numpy para las matrices,
# matplotlib para mostrar imágenes dentro del notebook.
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline


In [ ]:
# Carpeta donde viven todas las imágenes de la unidad.
IMG_DIR = os.path.join("assets", "img")


> 🔧 **OpenCV — `cv2.cvtColor(img, código)`**
> Convierte una imagen de un espacio de color a otro. `código` es una
> constante `cv2.COLOR_<ORIGEN>2<DESTINO>`, por ejemplo
> `cv2.COLOR_BGR2GRAY` (a escala de grises) o `cv2.COLOR_BGR2RGB` (para
> mostrar con matplotlib, como se ve dos celdas abajo). Devuelve un
> array nuevo — no modifica el original.

In [ ]:
def show(img, title="", figsize=(5, 4)):
    """Muestra una sola imagen: 1 canal -> gris, 3 canales -> BGR a RGB."""
    plt.figure(figsize=figsize)
    if img.ndim == 2:
        plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=10)
    plt.axis("off")
    plt.show()


In [ ]:
def show_grid(images_titles, cols=3, figsize=(4, 3.2), suptitle=None):
    """Muestra varias imágenes [(img, title), ...] en una grilla."""
    n = len(images_titles)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(figsize[0] * cols, figsize[1] * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (img, title) in zip(axes, images_titles):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


> 🔧 **OpenCV — `cv2.normalize(src, dst, alpha, beta, norm_type)`**
> Reescala los valores de un array al rango que quieras. Con
> `norm_type=cv2.NORM_MINMAX`, `alpha=0`, `beta=255`: el valor mínimo del
> array pasa a ser 0, el máximo pasa a ser 255, y todo lo demás se
> reparte proporcionalmente en el medio. `dst=None` le dice "créame un
> array de salida nuevo". Lo necesitamos constantemente en este notebook
> porque Sobel y Laplaciano devuelven números que pueden ser negativos o
> mayores a 255 — `norm8` los "aplana" a algo que se pueda dibujar.

In [ ]:
def norm8(mat):
    """Lleva cualquier rango (incluye negativos) a 0..255 uint8 para poder verlo."""
    return cv2.normalize(mat, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


### 0.2 · `cv2.imread`: leer una imagen del disco

Con las herramientas listas, cargamos las dos "cámaras" del reto.

> 🔧 **OpenCV — `cv2.imread(ruta, modo)`**
> Lee un archivo de imagen (PNG, JPG, …) y devuelve un array de NumPy —
> o `None` si no encuentra el archivo (nunca lanza una excepción, por
> eso siempre conviene un `assert ... is not None` justo después, como
> abajo). El parámetro `modo` decide la forma del array resultante:
>
> | modo | forma del array | uso típico |
> |---|---|---|
> | `cv2.IMREAD_COLOR` | `(alto, ancho, 3)`, en **BGR** | fotos a color |
> | `cv2.IMREAD_GRAYSCALE` | `(alto, ancho)`, 1 canal | casi todos los filtros de esta clase |
> | `cv2.IMREAD_UNCHANGED` | como esté el archivo (incluye alfa si tiene) | pocas veces hace falta |
>
> Aquí cargamos primero a color (para dibujar contornos en rojo más
> adelante) y sacamos la versión en gris con `cv2.cvtColor`, en vez de
> leer el archivo dos veces — es más barato convertir en memoria que
> volver a tocar disco.

In [ ]:
# Imagen 1: la foto real -- nuestra "cámara de cancha".
messi_color = cv2.imread(os.path.join(IMG_DIR, "image_messi.png"), cv2.IMREAD_COLOR)
assert messi_color is not None, "no se encontró image_messi.png"
messi_gray = cv2.cvtColor(messi_color, cv2.COLOR_BGR2GRAY)


In [ ]:
# Imagen 2: la escena sintética -- nuestro control.
escena_color = cv2.imread(os.path.join(IMG_DIR, "escena.png"), cv2.IMREAD_COLOR)
assert escena_color is not None, "no se encontró escena.png"
escena_gray = cv2.cvtColor(escena_color, cv2.COLOR_BGR2GRAY)


In [ ]:
print("messi :", messi_color.shape)
print("escena:", escena_color.shape)


`(alto, ancho, 3)` — el `.shape` de una imagen a color en OpenCV
siempre viene en ese orden (filas × columnas × canales), **no**
`(ancho, alto, ...)`. Es fácil confundirlo con coordenadas (x, y): en
`img.shape` el primer número es alto (`y`), no ancho (`x`).

In [ ]:
show_grid([
    (messi_color, "messi — color"),
    (messi_gray, "messi — gris"),
    (escena_color, "escena — color"),
    (escena_gray, "escena — gris"),
], cols=4)


## Acto 1 · Aritmética de imágenes

**SUMAR.**

Antes de vecindarios y kernels: lo más simple. Un píxel es un número, y
los números se pueden sumar, restar y mezclar. Esta es la operación más
básica de todas — sirve para combinar dos imágenes *completas*, no un
píxel con sus vecinos (eso es la convolución, y llega en el Acto 2).

### 1.1 · Sumar dos imágenes, píxel a píxel

Dos imágenes del mismo tamaño y del mismo tipo de dato: se suma el
valor de cada píxel con el píxel que está en la misma posición de la
otra imagen. Nada de vecinos — cada píxel solo mira a su pareja, en la
otra imagen.

Como `messi` y `escena` no miden lo mismo, primero hay que ponerlas al
mismo tamaño.

In [ ]:
# cv2.resize necesita (ancho, alto) -- al revés que .shape (alto, ancho).
alto, ancho = messi_gray.shape
escena_mismo_tamano = cv2.resize(escena_gray, (ancho, alto))

print("messi :", messi_gray.shape)
print("escena (redimensionada):", escena_mismo_tamano.shape)


> 🔧 **OpenCV — `cv2.add(a, b)`**
> Suma dos arrays del mismo tamaño y tipo, elemento a elemento — y
> **satura**: si el resultado pasa de 255, lo deja en 255 en vez de dar
> la vuelta. Es la versión segura de `a + b`.

In [ ]:
suma = cv2.add(messi_gray, escena_mismo_tamano)
show_grid([
    (messi_gray, "messi"),
    (escena_mismo_tamano, "escena (resize)"),
    (suma, "cv2.add(messi, escena)"),
], cols=3)


### 1.2 · El problema: el techo de 255

`uint8` guarda del 0 al 255. ¿Qué pasa si la suma da más que eso? Antes
de mirarlo sobre una imagen completa, probemos con un solo píxel.

In [ ]:
a = np.uint8([200])
b = np.uint8([100])

seguro = cv2.add(a, b)   # cv2 satura
peligroso = a + b        # numpy: "da la vuelta" (overflow)

print("cv2.add(200, 100)  ->", seguro[0])
print("200 + 100 a mano   ->", peligroso[0], " (300 mod 256)")


In [ ]:
print("""
OpenCV satura: nunca pasa de 255 ni baja de 0. Si sumas los arrays de
NumPy directamente, sin pasar por cv2.add, el tipo uint8 'da la vuelta'
-- 300 se convierte en 44. Un pixel que debia verse blanco y brillante,
se ve oscuro. Por eso: siempre cv2.add / cv2.subtract, nunca arr1+arr2
a mano cuando el resultado puede salirse de 0..255.
""")


### 1.3 · Resta y diferencia: ¿qué cambió?

Comparar dos cuadros seguidos de una cámara IoT, píxel a píxel, es la
forma más simple de detectar movimiento. Simulamos un "segundo cuadro":
la misma foto, con un rectángulo dibujado encima — algo que "apareció"
en la escena.

In [ ]:
cuadro1 = messi_gray.copy()
cuadro2 = messi_gray.copy()
cv2.rectangle(cuadro2, (60, 60), (160, 160), 255, thickness=-1)  # "algo" que entra al cuadro

show_grid([(cuadro1, "cuadro 1"), (cuadro2, "cuadro 2 (con algo nuevo)")], cols=2)


> 🔧 **OpenCV — `cv2.absdiff(a, b)`**
> Da directo el valor absoluto de la diferencia, `|a − b|`, sin importar
> cuál imagen es más clara — a diferencia de `cv2.subtract(a, b)`, que
> sí satura en 0 y sí depende de cuál va primero.

In [ ]:
diferencia = cv2.absdiff(cuadro1, cuadro2)
show(diferencia, "cv2.absdiff: aquí es donde cambió algo")


### 1.4 · Mezclar dos imágenes, con pesos

`resultado = α·A + β·B + γ`. Sirve para transiciones suaves entre dos
imágenes, una marca de agua semitransparente, o superponer un mapa de
calor sobre una foto.

> 🔧 **OpenCV — `cv2.addWeighted(a, α, b, β, γ)`**
> Combina dos imágenes con pesos. `α` y `β` no tienen que sumar 1 — pero
> si suman 1, el brillo total de la imagen se mantiene parecido.

In [ ]:
mezcla_70_30 = cv2.addWeighted(messi_gray, 0.7, escena_mismo_tamano, 0.3, 0)
mezcla_50_50 = cv2.addWeighted(messi_gray, 0.5, escena_mismo_tamano, 0.5, 0)

show_grid([
    (mezcla_70_30, "70% messi + 30% escena"),
    (mezcla_50_50, "50% / 50%"),
], cols=2)


### 1.5 · Aclarar y oscurecer: transformaciones logarítmica y exponencial

Sumar y restar cambian el brillo de forma pareja: le suman lo mismo a
un píxel oscuro que a uno claro. Para aclarar SOLO las sombras, o
oscurecer SOLO las luces altas, hace falta algo distinto: una curva, no
una suma.

In [ ]:
def transformacion_log(gray):
    """s = c * log(1 + r): aclara las sombras y comprime el rango dinámico."""
    r = gray.astype(np.float64)
    c = 255 / np.log(1 + 255)  # para que un 255 de entrada siga dando ~255
    s = c * np.log(1 + r)
    return np.uint8(s)

messi_log = transformacion_log(messi_gray)
show_grid([(messi_gray, "original"), (messi_log, "transformación logarítmica")], cols=2)


El logaritmo crece rápido al principio y luego se aplana: separa mejor
los tonos oscuros (donde antes casi no había diferencia entre un 5 y un
15) y aplasta un poco los tonos claros (donde 200 y 250 casi terminan
pareciéndose). Útil para "sacar" detalle de una foto subexpuesta — como
la de una cámara IoT filmando de noche.

In [ ]:
def ajustar_gamma(gray, gamma):
    """s = (r/255)^gamma * 255. gamma < 1 aclara, gamma > 1 oscurece."""
    normalizado = gray.astype(np.float64) / 255.0
    corregido = np.power(normalizado, gamma)
    return np.uint8(corregido * 255)

messi_gamma_claro = ajustar_gamma(messi_gray, 0.4)   # gamma < 1 -> aclara
messi_gamma_oscuro = ajustar_gamma(messi_gray, 2.5)  # gamma > 1 -> oscurece

show_grid([
    (messi_gamma_claro, "gamma 0.4 (aclara)"),
    (messi_gray, "original (gamma 1.0)"),
    (messi_gamma_oscuro, "gamma 2.5 (oscurece)"),
], cols=3)


In [ ]:
# El mismo píxel, visto por las dos curvas -- para comparar con números.
for valor in [50, 200]:
    r = valor / 255.0
    claro = int((r ** 0.4) * 255)
    oscuro = int((r ** 2.5) * 255)
    print(f"píxel {valor:3d}  ->  gamma 0.4: {claro:3d}   gamma 2.5: {oscuro:3d}")


> 🔧 **OpenCV — `cv2.pow(src, potencia)` / `cv2.log(src)`**
> OpenCV trae versiones dedicadas (más rápidas sobre arrays grandes),
> pero solo trabajan con `float32`/`float64`, no directo sobre `uint8` —
> por eso casi siempre se usa NumPy para esto: es igual de claro y no
> hay que preocuparse por el tipo de dato en cada paso intermedio.

Con esto ya sabemos combinar imágenes completas, y aclarar u oscurecer
sin perder la posición de cada píxel. Ahora, a combinar un píxel con
sus propios vecinos — eso es la convolución.

## Acto 2 · La operación

**VECINDARIO.**

Ya sumamos, restamos y mezclamos imágenes completas, número contra
número. Ahora algo distinto: un píxel solo casi no dice nada — lo
interesante aparece al compararlo con los píxeles que tiene alrededor,
dentro de la MISMA imagen. Todo lo que sigue en este notebook, y todo
lo que necesita el reto de la cámara de cancha, se resuelve con una
sola operación —la **convolución**— aplicada con distintas matrices de
pesos. Si entendemos bien esta primera pieza, las otras dos son
variaciones sobre el mismo tema.

### 2.1 · Qué es una convolución

Recorrer la imagen píxel por píxel y, en cada uno, **reemplazar su valor
por una combinación de él y sus vecinos**.

Los "pesos" de esa combinación son una matriz pequeña llamada **kernel**
(o máscara, o filtro). Cambiar el kernel cambia el efecto — es lo único
que cambia entre un blur y un detector de bordes. El código que lo
ejecuta (`cv2.filter2D`) es siempre el mismo.

Spoiler de todo lo que viene: suavizar, realzar, encontrar bordes — cada
uno de esos "poderes" es exactamente esta misma frase, con un kernel
distinto.

### 2.2 · El kernel: una matriz de pesos

Casi siempre es de tamaño impar (3×3, 5×5, 7×7) para que tenga un centro
bien definido: el centro es el peso del propio píxel, el resto el de
cada vecino. Aquí unos cuantos kernels que vamos a usar más abajo —
fíjate que son solo números, no código complicado (todavía no llamamos
a ninguna función de OpenCV; los kernels son puro NumPy).

In [ ]:
# Kernel identidad: no cambia nada (el centro pesa 1, el resto 0).
k_identidad = np.array([[0, 0, 0],
                         [0, 1, 0],
                         [0, 0, 0]], dtype=np.float32)
print("identidad (suma de pesos =", k_identidad.sum(), "):\n", k_identidad)


In [ ]:
# Kernel de realce (sharpen): centro positivo, vecinos negativos.
k_realce = np.array([[ 0, -1,  0],
                      [-1,  5, -1],
                      [ 0, -1,  0]], dtype=np.float32)
print("realce (suma de pesos =", k_realce.sum(), "):\n", k_realce)


In [ ]:
# Kernel de promedio (box blur) 3x3: todos los vecinos pesan igual.
k_promedio = np.ones((3, 3), dtype=np.float32) / 9
print("promedio (suma de pesos =", round(k_promedio.sum(), 3), "):\n", k_promedio)


### 2.3 · En cada píxel: alinear, multiplicar, sumar

La ventana del kernel se coloca sobre un píxel. Se multiplica cada valor
del vecindario por el peso del kernel en esa posición, se suman los
productos, y ese número es el nuevo valor del píxel central. Luego la
ventana se corre una posición y se repite — sobre toda la imagen.

Antes de dejar que el código calcule por nosotros, juguemos al
detective: el ejemplo con números de la clase es un vecindario 3×3 donde
la mitad de arriba vale 90 y la de abajo vale 10 — un borde horizontal
bien marcado — convolucionado con el kernel de Sobel vertical (detecta
cambios de arriba hacia abajo). Predice mentalmente si el resultado va a
ser un número grande o cercano a cero, y luego comprueba.

In [ ]:
# El vecindario del ejemplo: mitad de arriba clara (90), mitad de abajo oscura (10).
vecindario = np.array([[90, 90, 90],
                        [90, 90, 90],
                        [10, 10, 10]], dtype=np.float64)

# Kernel de Sobel vertical (Sobel Y): detecta cambios de arriba hacia abajo.
kernel_sobel_y = np.array([[-1, -2, -1],
                            [ 0,  0,  0],
                            [ 1,  2,  1]], dtype=np.float64)


In [ ]:
# "Alinear, multiplicar, sumar": producto elemento a elemento y luego sumar todo.
productos = vecindario * kernel_sobel_y
resultado = productos.sum()


In [ ]:
print("vecindario:\n", vecindario)
print("\nkernel (Sobel vertical / Sobel Y):\n", kernel_sobel_y)
print("\nproductos elemento a elemento:\n", productos)
print("\nsuma total (nuevo valor del píxel central):", resultado)


In [ ]:
# Verificación: (10+10+10) de abajo pesan +1/+2/+1, (90+90+90) de arriba pesan -1/-2/-1.
esperado = (10*1 + 10*2 + 10*1) - (90*1 + 90*2 + 90*1)
print("cálculo manual esperado:", esperado)
assert resultado == esperado, "no coincide con el cálculo manual"
print("OK: coincide.")


In [ ]:
print("""
Valor grande en magnitud (-320) = cambio brusco de arriba a abajo = hay
un borde horizontal aquí. En una zona plana, con todos los valores
parecidos, el resultado da ~0 (pruébalo cambiando 'vecindario' arriba a
un bloque uniforme, p. ej. np.full((3,3), 90)).
""")


### 2.4 · Implementar la convolución a mano

Antes de confiarle a `cv2.filter2D` el trabajo real —encontrar bordes
sobre la foto de la cancha—, construyámoslo nosotros mismos, para que no
sea una caja negra. Si `cv2.filter2D` no es más que este loop hecho más
rápido, entonces entender el loop es entender el filtro.

**Por qué en un recorte pequeño y no en toda la imagen:** un loop en
Python puro es ~100-1000× más lento que el mismo cálculo vectorizado en
C++ (que es lo que hace OpenCV por dentro). Sobre una imagen completa
tardaría segundos o minutos; sobre un recorte de 80×80 es instantáneo y
sirve igual para explicar y validar el mecanismo.

> 🔧 **OpenCV — `cv2.copyMakeBorder(img, arriba, abajo, izq, der, tipo, value=0)`**
> Devuelve una copia de `img` con un marco añadido alrededor, del ancho
> que le pidas en cada lado. `tipo` decide cómo se rellena ese marco
> (ver 2.5, un poco más abajo). Lo necesitamos aquí porque la ventana
> del kernel tiene que "caber" incluso en los píxeles del borde de la
> imagen — sin este relleno, no habría vecinos que leer ahí.

In [ ]:
def convolucion_manual(img, kernel, padding="reflect"):
    """
    Convolución 2D de referencia, sin cv2.filter2D, para ver el mecanismo
    'alinear, multiplicar, sumar' explícitamente. Solo para explicar/validar
    -- para uso real, siempre cv2.filter2D (o la función dedicada del filtro).
    """
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    borde = {
        "reflect": cv2.BORDER_REFLECT101,   # espejo hacia adentro (default de OpenCV)
        "replicate": cv2.BORDER_REPLICATE,  # repite el último píxel del borde
        "constant": cv2.BORDER_CONSTANT,    # rellena con un valor fijo (0 por defecto)
    }[padding]

    padded = cv2.copyMakeBorder(img.astype(np.float64), pad_h, pad_h, pad_w, pad_w, borde)
    salida = np.zeros_like(img, dtype=np.float64)

    h, w = img.shape
    for y in range(h):
        for x in range(w):
            ventana = padded[y:y + kh, x:x + kw]      # alinear
            salida[y, x] = np.sum(ventana * kernel)   # multiplicar + sumar
    return salida


In [ ]:
# Recorte pequeño de la foto real para que el loop sea instantáneo.
recorte = messi_gray[100:180, 100:180]
print("recorte:", recorte.shape)


In [ ]:
manual = convolucion_manual(recorte, k_realce, padding="reflect")


In [ ]:
show_grid([(recorte, "recorte original"), (norm8(manual), "convolución manual (realce)")], cols=2)


### 2.5 · ¿Y en el borde de la imagen?

En la primera fila y la primera columna, la ventana del kernel **se sale
de la imagen** — un problema muy real si la cámara enmarca al jugador
pegado a una esquina del cuadro. Hay que rellenar lo que falta con algún
criterio de padding antes de convolucionar. Tres formas típicas — se
nota la diferencia sobre todo en `constant`, que mete un marco oscuro
artificial:

In [ ]:
pad = 40  # exagerado a propósito para que la diferencia se vea clara
tipos = {
    "reflect (BORDER_REFLECT101)": cv2.BORDER_REFLECT101,
    "replicate (BORDER_REPLICATE)": cv2.BORDER_REPLICATE,
    "constant=0 (BORDER_CONSTANT)": cv2.BORDER_CONSTANT,
}


In [ ]:
paneles = []
for nombre, tipo in tipos.items():
    bordeado = cv2.copyMakeBorder(recorte, pad, pad, pad, pad, tipo, value=0)
    paneles.append((bordeado, nombre))


In [ ]:
show_grid(paneles, cols=3)
print("Sin este relleno, cv2.filter2D no podría calcular los píxeles del borde.")


### 2.6 · Validar contra `cv2.filter2D`

El momento de la verdad. Si nuestra implementación manual es correcta,
debe dar **exactamente** lo mismo que la función de OpenCV, píxel por
píxel (salvo redondeos de punto flotante). Si no coincide, algo en
nuestro padding o en nuestro kernel está mal — y es mejor descubrirlo
aquí, sobre un recorte de 80×80, que mucho más adelante sobre la imagen
completa de la cancha.

> 🔧 **OpenCV — `cv2.filter2D(img, ddepth, kernel, borderType=...)`**
> La convolución genérica de OpenCV: aplica *cualquier* kernel 2D a una
> imagen. `ddepth` es la profundidad (el tipo de dato) de la salida —
> `-1` significa "la misma que la de entrada". Vas a ver este mismo
> parámetro `ddepth` en Sobel y Laplaciano más adelante (Acto 4): ahí sí
> importa ponerlo distinto de `-1`, y ya vas a saber por qué.

In [ ]:
oficial = cv2.filter2D(recorte.astype(np.float64), -1, k_realce, borderType=cv2.BORDER_REFLECT101)
diferencia_maxima = np.max(np.abs(manual - oficial))
print("diferencia máxima entre manual y cv2.filter2D:", diferencia_maxima)


In [ ]:
assert np.allclose(manual, oficial), "no coinciden -- revisar padding o kernel"
print("OK: cv2.filter2D hace exactamente lo mismo que nuestro loop, pero vectorizado.")


### 2.7 · `cv2.filter2D` en la práctica: el realce aplicado

Con la validación hecha, ya podemos confiar en la versión rápida sobre
las imágenes completas. El kernel de realce acentúa los bordes: donde
había un cambio suave, ahora hay un salto más marcado.

**Detalle importante:** la suma de los pesos del kernel de realce es 1,
así que el brillo promedio de la imagen *no debería cambiar* — solo el
contraste local. Lo comprobamos comparando el promedio de grises antes y
después.

Con esto ya tenemos la herramienta base. En teoría, podríamos ir directo
a buscar bordes y cerrar el reto aquí mismo. Pero hay un problema que
estuvimos evitando a propósito: la cámara barata de la cancha mete
ruido, y para un detector de bordes el ruido se ve **exactamente** como
un borde real. Eso es el Acto 3.

In [ ]:
messi_realce = cv2.filter2D(messi_gray, -1, k_realce)
escena_realce = cv2.filter2D(escena_gray, -1, k_realce)


In [ ]:
show_grid([
    (messi_gray, "messi — original"),
    (messi_realce, "messi — filter2D(realce)"),
    (escena_gray, "escena — original"),
    (escena_realce, "escena — filter2D(realce)"),
], cols=2)


In [ ]:
print(f"brillo medio messi:  original={messi_gray.mean():.1f}  realzado={messi_realce.mean():.1f}")
print(f"brillo medio escena: original={escena_gray.mean():.1f}  realzado={escena_realce.mean():.1f}")


## Acto 3 · Suavizado

**SUAVIZAR.**

La cámara de cancha no es un sensor de laboratorio: poca luz, ISO alto,
compresión de video — todo eso mete variación aleatoria en cada cuadro.
Si el Acto 4 (detectar bordes) corriera directo sobre esto, "vería" un
borde en cada píxel ruidoso, y el contorno del balón se perdería entre
cientos de bordes falsos.

Este acto resuelve eso: reemplazar cada píxel por un promedio (o algo
parecido) de su vecindario, para borrar el ruido sin borrar el balón.
También aparecen las tres primeras funciones de OpenCV "dedicadas" — más
rápidas y con mejores valores por defecto que llamar a `filter2D` a
mano.

### 3.1 · El problema: ruido

Poca luz, ISO alto, sensor barato: los valores de píxeles que deberían
ser iguales llegan con variaciones aleatorias. Si buscamos bordes sobre
esto directamente (Acto 4), encontramos cientos de bordes falsos — por
eso el suavizado va *antes* de detectar bordes, no después.

Antes de limpiar, hay que ensuciar a propósito: simulamos el tipo de
ruido que mete una cámara barata (gaussiano por el sensor + sal y
pimienta por transmisión/compresión) sobre las dos imágenes, para tener
algo real que limpiar en el resto del acto. Esta función es puro
NumPy — todavía no es OpenCV, es la "cámara mala" que vamos a limpiar
con OpenCV después.

In [ ]:
def agregar_ruido(gray, seed):
    """Ruido gaussiano (sensor) + sal y pimienta (transmisión/compresión)."""
    rng = np.random.default_rng(seed)
    ruido_gauss = rng.normal(0, 22, gray.shape)
    con_ruido = np.clip(gray.astype(np.float64) + ruido_gauss, 0, 255).astype(np.uint8)

    sal_pimienta = rng.random(gray.shape)
    con_ruido[sal_pimienta < 0.02] = 0     # "pimienta": puntos negros
    con_ruido[sal_pimienta > 0.98] = 255   # "sal": puntos blancos
    return con_ruido


In [ ]:
messi_ruido = agregar_ruido(messi_gray, seed=7)
escena_ruido = agregar_ruido(escena_gray, seed=7)


In [ ]:
show_grid([
    (messi_gray, "messi — sin ruido"),
    (messi_ruido, "messi — con ruido"),
    (escena_gray, "escena — sin ruido"),
    (escena_ruido, "escena — con ruido"),
], cols=2)


### 3.2 · Blur de caja (promedio)

Kernel de puros unos dividido entre el número de celdas: se suman los
vecinos y se divide entre 9 (para 3×3). Todos los vecinos pesan igual.
Kernel más grande (5×5, 9×9…) = más suavizado. Sencillo y rápido, pero
"emborrona" todo por igual y puede dejar cajas visibles — nuestro primer
candidato para limpiar la cámara de cancha, aunque no el mejor, como
vamos a ver.

> 🔧 **OpenCV — `cv2.blur(img, (ancho, alto))`**
> Atajo para `filter2D` con el kernel de promedio ya armado — evita que
> tengas que escribir `np.ones((9,9))/81` a mano cada vez. El tamaño de
> ventana debe ser una tupla de dos enteros, casi siempre impares.

In [ ]:
k_box_3 = np.ones((3, 3), np.float32) / 9
print("kernel de caja 3x3 (cada celda pesa 1/9):\n", k_box_3)


In [ ]:
messi_box = cv2.blur(messi_ruido, (9, 9))
escena_box = cv2.blur(escena_ruido, (9, 9))


In [ ]:
show_grid([(messi_ruido, "con ruido"), (messi_box, "blur caja 9x9")], cols=2)


### 3.3 · Blur gaussiano

Los pesos siguen una campana de Gauss: el píxel central y sus vecinos
cercanos importan más que los lejanos. Resultado más natural que el
promedio, sin "cajas". Es el suavizado estándar antes de detectar
bordes — el que vamos a usar en el resto del notebook.

> 🔧 **OpenCV — `cv2.GaussianBlur(img, (ancho, alto), sigma)`**
> El tamaño de ventana, igual que en `cv2.blur`, debe ser impar.
> `sigma` controla qué tan ancha es la campana (cuánto se suaviza); si
> le pasas `0`, OpenCV lo calcula automáticamente a partir del tamaño de
> la ventana. `cv2.getGaussianKernel(n, sigma)`, usado abajo solo para
> imprimir los pesos, es la pieza interna que arma esa campana en 1D —
> no hace falta llamarla para usar `GaussianBlur`, es la misma función
> por dentro.

In [ ]:
# cv2.getGaussianKernel da el kernel 1D; el 2D es el producto exterior consigo mismo.
k1d = cv2.getGaussianKernel(9, 2.5)
k_gauss_2d = k1d @ k1d.T
print("kernel gaussiano 9x9 (sigma=2.5), redondeado a 3 decimales:")
print(np.round(k_gauss_2d, 3))
print("suma de los pesos (debe ser ~1):", k_gauss_2d.sum())


In [ ]:
messi_gauss = cv2.GaussianBlur(messi_ruido, (9, 9), 2.5)
escena_gauss = cv2.GaussianBlur(escena_ruido, (9, 9), 2.5)


In [ ]:
show_grid([(messi_ruido, "con ruido"), (messi_gauss, "blur gaussiano 9x9")], cols=2)


### 3.4 · Blur de mediana

La mediana **no es una convolución**: en vez de una suma ponderada, toma
el valor mediano del vecindario (no hay kernel de pesos). Por eso
elimina el ruido de sal y pimienta casi por completo, mientras que
promedio y gaussiano solo lo atenúan (un solo píxel a 0 o 255 arrastra
el promedio, pero casi nunca es la mediana). Es la herramienta correcta
para el tipo de ruido "puntos sueltos" que deja la compresión de video
en un sistema IoT.

> 🔧 **OpenCV — `cv2.medianBlur(img, tamaño)`**
> Aquí `tamaño` es un solo entero impar (no una tupla, a diferencia de
> `blur` y `GaussianBlur`), porque la ventana de la mediana siempre es
> cuadrada.

In [ ]:
messi_mediana = cv2.medianBlur(messi_ruido, 5)
escena_mediana = cv2.medianBlur(escena_ruido, 5)


In [ ]:
show_grid([(messi_ruido, "con ruido"), (messi_mediana, "blur mediana 5x5")], cols=2)


### 3.5 · Los tres, lado a lado, sobre ambas imágenes

Comparación directa: promedio vs. gaussiano vs. mediana, misma imagen
con ruido, mismo tamaño de ventana aproximado. ¿Cuál dejarías corriendo
en la Raspberry Pi de la cancha?

In [ ]:
show_grid([
    (messi_ruido, "messi — con ruido"),
    (messi_box, "promedio 9x9 (cv2.blur)"),
    (messi_gauss, "gaussiano 9x9 (cv2.GaussianBlur)"),
    (messi_mediana, "mediana 5x5 (cv2.medianBlur)"),
], cols=4, suptitle="messi")


In [ ]:
show_grid([
    (escena_ruido, "escena — con ruido"),
    (escena_box, "promedio 9x9"),
    (escena_gauss, "gaussiano 9x9"),
    (escena_mediana, "mediana 5x5"),
], cols=4, suptitle="escena")


### 3.6 · ¿Cuál sirve mejor contra cada tipo de ruido?

En vez de solo mirar las imágenes, lo medimos: cuánto "sal y pimienta"
queda (píxeles casi en 0 o casi en 255) después de cada filtro.

**Regla práctica:** casi siempre `GaussianBlur`. La mediana, cuando el
ruido es de puntos aislados (sal y pimienta). El promedio, casi nunca —
el gaussiano da mejor resultado por el mismo costo.

Con esto elegimos `GaussianBlur` para limpiar el resto del recorrido. La
imagen ya está limpia — pero "limpia" no es lo mismo que "con los
bordes marcados". Para eso, el Acto 4.

In [ ]:
def contar_extremos(img, umbral=15):
    """Cuenta píxeles casi-negros o casi-blancos: rastro de sal y pimienta."""
    return int(np.sum((img < umbral) | (img > 255 - umbral)))


In [ ]:
for nombre, img in [("con ruido", messi_ruido), ("promedio", messi_box),
                     ("gaussiano", messi_gauss), ("mediana", messi_mediana)]:
    print(f"{nombre:12s} -> píxeles extremos: {contar_extremos(img):5d}")


## Acto 4 · Bordes

**BORDES.**

Ya tenemos una imagen limpia. Ahora sí: encontrar dónde está el borde
del balón, del jugador, de la cancha. Un borde es un sitio donde la
intensidad cambia de golpe — encontrarlo es, literalmente, medir un
cambio: una derivada.

### 4.1 · Un borde es una derivada grande

Antes de aplicar Sobel en 2D, veámoslo en 1D: tomamos una sola fila (o
columna) de píxeles y calculamos la diferencia entre cada valor y el
siguiente (`np.diff`, que es exactamente el kernel `[-1, 0, 1]` de la
clase). Donde la imagen es plana, la diferencia es ~0. Donde hay un
borde, hay un pico — ese pico es, en miniatura, lo que vamos a buscar en
toda la imagen con Sobel. (`np.diff` es de NumPy, no de OpenCV — lo
usamos aquí precisamente para ver la idea sin la maquinaria de OpenCV de
por medio, antes de usar `cv2.Sobel` en la siguiente sección.)

Elegimos automáticamente la fila con más variación total, para
garantizar que se vea un borde real sin importar el contenido exacto de
cada imagen.

In [ ]:
def fila_mas_variable(gray):
    """Índice de la fila con mayor cambio total (para asegurar que cruce un borde)."""
    variacion_por_fila = np.sum(np.abs(np.diff(gray.astype(np.int16), axis=1)), axis=1)
    return int(np.argmax(variacion_por_fila))


In [ ]:
fila = fila_mas_variable(escena_gray)
perfil = escena_gray[fila, :].astype(np.int16)
derivada = np.diff(perfil)  # kernel [-1, 0, 1] en 1D: derecha - izquierda


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(perfil)
axes[0].set_title(f"escena — intensidad a lo largo de la fila {fila}")
axes[0].set_ylabel("valor de gris")
axes[1].plot(derivada, color="tab:red")
axes[1].set_title("derivada (np.diff): los picos son los bordes")
axes[1].set_xlabel("posición x")
axes[1].axhline(0, color="black", linewidth=0.6)
plt.tight_layout()
plt.show()


### 4.2 · Sobel: derivada en x y en y

Es la derivada `[-1, 0, 1]` con un pequeño suavizado perpendicular
(`[1, 2, 1]`) que la hace menos sensible al ruido puntual. Sobel X
responde a cambios en horizontal (contornos verticales); Sobel Y, al
revés (contornos horizontales).

In [ ]:
k_sobel_x = np.array([[-1, 0, 1],
                       [-2, 0, 2],
                       [-1, 0, 1]], dtype=np.float32)
print("Sobel X (bordes verticales):\n", k_sobel_x)


In [ ]:
k_sobel_y = np.array([[-1, -2, -1],
                       [ 0,  0,  0],
                       [ 1,  2,  1]], dtype=np.float32)
print("Sobel Y (bordes horizontales):\n", k_sobel_y)


### 4.3 · Aplicar Sobel: X, Y y la magnitud

**Por qué `CV_64F` y no `CV_8U`:** un borde puede ir "de claro a oscuro"
o "de oscuro a claro", así que la derivada puede dar negativa. Si la
salida se guarda en `uint8` (0..255), OpenCV recorta los negativos a 0 y
se pierde la mitad de los bordes. Por eso se calcula en `CV_64F` (permite
negativos) y se normaliza al final solo para poder visualizarlo.

En el reto de la cancha, un borde perdido aquí es literalmente un
pedazo del contorno del balón que el sistema nunca va a detectar.
Siempre se suaviza antes con `GaussianBlur` — si no, el ruido genera
gradientes falsos en todas partes (ver 3.1).

> 🔧 **OpenCV — `cv2.Sobel(img, ddepth, dx, dy, ksize=3)`**
> `ddepth` es el mismo parámetro que ya viste en `filter2D` — aquí lo
> ponemos en `cv2.CV_64F` (float de 64 bits) en vez de `-1`, justo para
> no perder los negativos. `dx`, `dy` dicen en qué dirección derivar:
> `(1, 0)` = derivada en x (Sobel X), `(0, 1)` = derivada en y (Sobel Y).
> `ksize` es el tamaño del kernel de Sobel (3 = el 3×3 de arriba).
>
> 🔧 **OpenCV — `cv2.magnitude(x, y)`**
> Calcula `sqrt(x**2 + y**2)` elemento a elemento — podríamos escribirlo
> con NumPy puro, pero la versión de OpenCV está optimizada para esto.

In [ ]:
def sobel_xy_mag(gray):
    suave = cv2.GaussianBlur(gray, (3, 3), 0)      # 1. siempre suavizar antes
    gx = cv2.Sobel(suave, cv2.CV_64F, 1, 0, ksize=3)  # 2. derivada en x
    gy = cv2.Sobel(suave, cv2.CV_64F, 0, 1, ksize=3)  # 3. derivada en y
    mag = cv2.magnitude(gx, gy)                       # 4. sqrt(gx^2 + gy^2)
    return gx, gy, mag


In [ ]:
messi_gx, messi_gy, messi_mag = sobel_xy_mag(messi_gray)
escena_gx, escena_gy, escena_mag = sobel_xy_mag(escena_gray)


In [ ]:
show_grid([
    (norm8(np.abs(messi_gx)), "|Sobel X| — bordes verticales"),
    (norm8(np.abs(messi_gy)), "|Sobel Y| — bordes horizontales"),
    (norm8(messi_mag), "magnitud sqrt(Gx²+Gy²)"),
], cols=3, suptitle="messi")


In [ ]:
show_grid([
    (norm8(np.abs(escena_gx)), "|Sobel X|"),
    (norm8(np.abs(escena_gy)), "|Sobel Y| — nota el horizonte"),
    (norm8(escena_mag), "magnitud"),
], cols=3, suptitle="escena")


### 4.4 · La trampa: `CV_64F` vs `CV_8U`

Esta es la trampa en la que cae casi todo el mundo la primera vez que
usa Sobel: pedirle a OpenCV la salida directo en `uint8` (parece más
simple, "menos memoria"). Repetimos Sobel X pero con esa salida, y
comparamos cuántos bordes "desaparecen" por el recorte de negativos.

In [ ]:
suave = cv2.GaussianBlur(messi_gray, (3, 3), 0)
sobel_64f = cv2.Sobel(suave, cv2.CV_64F, 1, 0, ksize=3)       # correcto: sin recorte
sobel_8u  = cv2.Sobel(suave, cv2.CV_8U,  1, 0, ksize=3)       # incorrecto: recorta negativos a 0


In [ ]:
perdidos = int(np.sum(sobel_64f < 0))
total = sobel_64f.size
print(f"píxeles con derivada negativa: {perdidos} de {total} ({100*perdidos/total:.1f}%)")
print("-> todos esos bordes se pierden si se pide la salida directo en CV_8U.")


In [ ]:
show_grid([
    (norm8(np.abs(sobel_64f)), "CV_64F, normalizado (correcto)"),
    (sobel_8u, "CV_8U directo (pierde ~la mitad)"),
], cols=2)


### 4.5 · Laplaciano: un solo kernel, todas las orientaciones

Es la **segunda** derivada: mide la curvatura, no la pendiente. Detecta
bordes en cualquier dirección con un solo kernel (no hace falta X e Y
por separado), pero es muy sensible al ruido — casi obligatorio un
`GaussianBlur` antes (la combinación se llama **LoG**, *Laplacian of
Gaussian*). Lo mostramos con y sin ese suavizado previo para que se note
la diferencia: es el mismo problema del Acto 3, otra vez, con otro
filtro.

> 🔧 **OpenCV — `cv2.Laplacian(img, ddepth, ksize=3)`**
> Mismo patrón que `cv2.Sobel`: `ddepth=cv2.CV_64F` para no perder
> negativos, `ksize` el tamaño del kernel. No hay `dx`/`dy` porque el
> Laplaciano no tiene dirección — el kernel de la celda de abajo ya
> combina las cuatro direcciones en un solo número por píxel.

In [ ]:
k_laplaciano = np.array([[0,  1, 0],
                          [1, -4, 1],
                          [0,  1, 0]], dtype=np.float32)
print("kernel del Laplaciano:\n", k_laplaciano)


In [ ]:
suave = cv2.GaussianBlur(messi_gray, (3, 3), 0)
lap_con_blur = cv2.Laplacian(suave, cv2.CV_64F, ksize=3)
lap_sin_blur_con_ruido = cv2.Laplacian(messi_ruido, cv2.CV_64F, ksize=3)


In [ ]:
show_grid([
    (norm8(np.abs(lap_con_blur)), "Laplaciano (con GaussianBlur antes)"),
    (norm8(np.abs(lap_sin_blur_con_ruido)), "Laplaciano (sobre imagen con ruido, sin suavizar)"),
], cols=2)
print("El segundo amplifica el ruido en vez de resaltar solo los bordes reales.")


### 4.6 · Canny no es un kernel: es un pipeline

Este es el filtro que de verdad va a resolver el reto. Cuatro pasos, no
una sola multiplicación de matriz:

1. **Suavizar** — `GaussianBlur` para quitar ruido.
2. **Gradiente** — Sobel X e Y → magnitud y dirección en cada píxel.
3. **Adelgazar** (*non-max suppression*) — dejar solo el píxel más fuerte
   a lo ancho de cada borde → líneas de 1 px.
4. **Histéresis** (dos umbrales) — borde seguro si supera el umbral alto;
   se conserva el débil solo si toca a uno seguro.

Por eso el resultado son líneas finas y continuas, no un mapa de
intensidades como Sobel o Laplaciano — justo lo que necesita
`cv2.findContours` para trazar el contorno del balón en el paso final.

> 🔧 **OpenCV — `cv2.Canny(img, umbral_bajo, umbral_alto)`**
> A diferencia de Sobel y Laplaciano, no recibe `ddepth`: el resultado
> siempre es `uint8` de un solo canal, con solo dos valores posibles por
> píxel — 0 (no es borde) o 255 (sí es borde). Los cuatro pasos de
> arriba ocurren todos dentro de esta única llamada.

In [ ]:
# umbral_bajo, umbral_alto -- regla práctica: alto ~= 2 a 3 veces el bajo
canny_balanceado = cv2.Canny(cv2.GaussianBlur(messi_gray, (5, 5), 0), 80, 160)


In [ ]:
# umbrales bajos, aplicados sobre la imagen CON ruido y sin suavizar -> bordes falsos
canny_bajo = cv2.Canny(messi_ruido, 20, 50)


In [ ]:
# umbrales altos -> solo sobreviven los bordes de mayor contraste
canny_alto = cv2.Canny(cv2.GaussianBlur(messi_gray, (5, 5), 0), 180, 300)


In [ ]:
show_grid([
    (canny_bajo, "umbrales bajos + ruido -> bordes falsos"),
    (canny_balanceado, "80 / 160 -> equilibrado"),
    (canny_alto, "umbrales altos -> solo bordes fuertes"),
], cols=3, suptitle="messi")


In [ ]:
for nombre, img in [("bajo+ruido", canny_bajo), ("balanceado", canny_balanceado), ("alto", canny_alto)]:
    print(f"{nombre:12s} -> píxeles de borde: {int(np.sum(img > 0)):6d}")


**Regla práctica de umbrales:** el umbral alto ≈ 2 a 3 veces el bajo.
Demasiados bordes falsos → subir umbrales. Faltan bordes reales → bajarlos.
Depende de la imagen y de la iluminación — no hay un valor universal.

Con Sobel, Laplaciano y Canny ya en la caja de herramientas, tenemos
todas las piezas para volver al reto original: encontrar el contorno del
balón (o de Messi) automáticamente, sobre una imagen real y ruidosa.

## Pipeline completo: de píxeles a contornos

**El reto, resuelto.** Leer → escala de grises → `GaussianBlur` (Acto 3)
→ `Canny` (Acto 4) → `cv2.findContours`. Cada paso es una pieza que ya
construimos y entendimos por separado; aquí solo las encadenamos. Esto
es exactamente lo que alimentaría la etapa de "procesamiento" del
pipeline AIoT de la Unidad 01, corriendo en la placa junto a la cámara
de la cancha. Lo corremos sobre las dos imágenes.

> 🔧 **OpenCV — `cv2.findContours(bordes, modo, método)`**
> Recibe una imagen binaria (como la salida de Canny: solo 0 y 255) y
> devuelve la lista de contornos que encuentra — cada contorno es un
> array de puntos (x, y) que bordean una forma. `modo=cv2.RETR_EXTERNAL`
> pide solo los contornos "de afuera" (ignora huecos internos);
> `método=cv2.CHAIN_APPROX_SIMPLE` comprime líneas rectas a solo sus dos
> extremos, en vez de guardar cada píxel de por medio.
>
> 🔧 **OpenCV — `cv2.drawContours(img, contornos, índice, color, grosor)`**
> Dibuja contornos **encima** de `img` (la modifica directamente — por
> eso usamos `.copy()` antes, para no dañar la imagen original).
> `índice=-1` significa "dibújalos todos"; `color` va en BGR, como
> siempre en OpenCV (por eso `(29, 35, 214)` es rojo, no azul).

In [ ]:
def pipeline_contornos(color_img, gray_img):
    bordes = cv2.Canny(cv2.GaussianBlur(gray_img, (5, 5), 0), 80, 160)
    contornos, _ = cv2.findContours(bordes, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    dibujo = color_img.copy()
    cv2.drawContours(dibujo, contornos, -1, (29, 35, 214), 2)  # rojo de la marca (BGR)
    return dibujo, contornos


In [ ]:
messi_contornos, cnts_messi = pipeline_contornos(messi_color, messi_gray)


In [ ]:
escena_contornos, cnts_escena = pipeline_contornos(escena_color, escena_gray)


In [ ]:
show_grid([
    (messi_contornos, f"messi — {len(cnts_messi)} contornos"),
    (escena_contornos, f"escena — {len(cnts_escena)} contornos"),
], cols=2, figsize=(5, 4))


## Epílogo: un filtro = un kernel = un efecto

Empezamos con aritmética simple entre imágenes completas. Después, la
promesa: que casi todo el procesamiento clásico de imágenes es la misma
operación con distinta matriz de pesos. Tras cuatro actos, la tabla de
abajo es esa promesa cumplida — siete filtros, un solo mecanismo
(`cv2.filter2D`, o su prima cercana en el caso de la mediana).

| Filtro | Qué hace el kernel | Efecto | OpenCV |
|---|---|---|---|
| Promedio | Media del vecindario (pesos iguales) | Suaviza, quita detalle y ruido | `cv2.blur` |
| Gaussiano | Media ponderada por campana de Gauss | Suaviza natural; paso previo a bordes | `cv2.GaussianBlur` |
| Mediana | Valor mediano (no es convolución) | Elimina sal y pimienta | `cv2.medianBlur` |
| Realce | Centro positivo, vecinos negativos | Acentúa los bordes existentes | `cv2.filter2D` |
| Sobel | Primera derivada en x o en y | Fuerza y dirección de los bordes | `cv2.Sobel` |
| Laplaciano | Segunda derivada (todas direcciones) | Bordes finos; sensible al ruido | `cv2.Laplacian` |
| Canny | Pipeline: blur → Sobel → adelgazar → histéresis | Líneas de borde finas y continuas | `cv2.Canny` |

Y la prueba final, de un vistazo: todos los filtros, uno al lado del
otro, sobre el mismo recorte de `messi` — el mismo pedazo de cancha,
visto con cada herramienta de la caja.

In [ ]:
recorte_color = messi_color[80:260, 80:280]
recorte_gray = cv2.cvtColor(recorte_color, cv2.COLOR_BGR2GRAY)
recorte_con_ruido = agregar_ruido(recorte_gray, seed=3)


In [ ]:
panel_resumen = [
    (recorte_gray, "original"),
    (cv2.filter2D(recorte_gray, -1, k_realce), "realce (filter2D)"),
    (cv2.blur(recorte_con_ruido, (9, 9)), "promedio (blur)"),
    (cv2.GaussianBlur(recorte_con_ruido, (9, 9), 2.5), "gaussiano"),
    (cv2.medianBlur(recorte_con_ruido, 5), "mediana"),
    (norm8(np.abs(cv2.Sobel(recorte_gray, cv2.CV_64F, 1, 0, ksize=3))), "Sobel X"),
    (norm8(np.abs(cv2.Laplacian(cv2.GaussianBlur(recorte_gray, (3, 3), 0), cv2.CV_64F))), "Laplaciano"),
    (cv2.Canny(cv2.GaussianBlur(recorte_gray, (5, 5), 0), 80, 160), "Canny"),
]


In [ ]:
show_grid(panel_resumen, cols=4, figsize=(3.6, 3.2), suptitle="Un filtro = un kernel = un efecto")


## Apéndice · Chuleta de funciones de OpenCV

Todas las funciones de `cv2` que usamos en este notebook, para
consultar rápido sin tener que buscarlas en cada sección:

| Función | Para qué sirve |
|---|---|
| `cv2.imread(ruta, modo)` | Leer una imagen del disco a un array de NumPy (BGR si es a color) |
| `cv2.cvtColor(img, código)` | Convertir entre espacios de color (`BGR2GRAY`, `BGR2RGB`, …) |
| `cv2.resize(img, (ancho, alto))` | Cambiar el tamaño de una imagen — necesario antes de operar dos imágenes de tamaños distintos |
| `cv2.add(a, b)` / `cv2.subtract(a, b)` | Sumar o restar dos imágenes, con saturación (nunca pasan de 255 ni bajan de 0) |
| `cv2.absdiff(a, b)` | Diferencia absoluta `\|a − b\|`, sin importar cuál es mayor |
| `cv2.addWeighted(a, α, b, β, γ)` | Mezcla ponderada: `α·a + β·b + γ` |
| `cv2.normalize(src, dst, alpha, beta, tipo)` | Reescalar valores a un rango (p. ej. 0..255) |
| `cv2.copyMakeBorder(img, arriba, abajo, izq, der, tipo)` | Añadir un marco de relleno alrededor de la imagen |
| `cv2.filter2D(img, ddepth, kernel)` | Aplicar cualquier kernel de convolución |
| `cv2.blur(img, tamaño)` | Suavizado de caja (promedio) |
| `cv2.GaussianBlur(img, tamaño, sigma)` | Suavizado gaussiano |
| `cv2.getGaussianKernel(n, sigma)` | El kernel 1D que usa `GaussianBlur` por dentro |
| `cv2.medianBlur(img, tamaño)` | Suavizado de mediana (no es convolución) |
| `cv2.Sobel(img, ddepth, dx, dy, ksize)` | Derivada (gradiente) en x o en y |
| `cv2.magnitude(x, y)` | `sqrt(x² + y²)` elemento a elemento |
| `cv2.Laplacian(img, ddepth, ksize)` | Segunda derivada, todas las direcciones a la vez |
| `cv2.Canny(img, umbral_bajo, umbral_alto)` | Pipeline completo de detección de bordes |
| `cv2.findContours(bordes, modo, método)` | Extraer las formas (contornos) de una imagen binaria |
| `cv2.drawContours(img, contornos, índice, color, grosor)` | Dibujar contornos sobre una imagen |

**Los cuatro patrones que se repiten** una y otra vez en la API de
OpenCV, y que ya conoces después de este notebook:

1. **Saturación, no desbordamiento** (`add`, `subtract`, y en general
   cualquier operación aritmética entre imágenes): `uint8` nunca pasa de
   255 ni baja de 0 al usar funciones de `cv2` — pero sí "da la vuelta"
   si operas los arrays de NumPy a mano.
2. **`ddepth`** (`filter2D`, `Sobel`, `Laplacian`): el tipo de dato de
   salida. `-1` = igual que la entrada; `cv2.CV_64F` cuando el resultado
   puede ser negativo o salirse de 0..255.
3. **Tamaño de kernel/ventana impar** (`blur`, `GaussianBlur`,
   `medianBlur`, `Sobel`, `Laplacian`, `Canny` con `GaussianBlur` antes):
   para que exista un centro bien definido.
4. **BGR, no RGB** (`imread`, `cvtColor`, el color de `drawContours`):
   en todo lo que sea "color" en OpenCV, el orden de canales es azul,
   verde, rojo.